# Risk Parity Portfolio — Demo

This notebook demonstrates the `riskparity` package, which implements the **improved CCD method** from:

> Choi, J., & Chen, R. (2022). *Improved iterative methods for solving risk parity portfolio.*  
> Journal of Derivatives and Quantitative Studies, 30(2).  
> https://doi.org/10.1108/JDQS-12-2021-0031

A **risk parity** portfolio allocates weights so that every asset contributes equally to total portfolio volatility:

$$\frac{w_i (\mathbf{C}\mathbf{w})_i}{\mathbf{w}^\top \mathbf{C}\mathbf{w}} = \frac{1}{N} \quad \forall\, i$$

where $\mathbf{C}$ is the return covariance matrix.

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    import riskparity
except ImportError:
    %pip install mafn-pypackagetest --quiet
    import riskparity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from riskparity import risk_parity, risk_contributions

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

---
## 1. Two-asset example

A simple sanity check: two assets with different volatilities.

In [ ]:
# Covariance matrix: σ₁=20%, σ₂=30%, correlation=0.5
cov2 = np.array([[0.04, 0.03],
                 [0.03, 0.09]])

result2 = risk_parity(cov2)

print(f"Weights              : {result2.weights}")
print(f"Risk contributions   : {result2.risk_contributions}")
print(f"Iterations           : {result2.iterations}")
print(f"Converged            : {result2.converged}")

---
## 2. Five-asset equal risk parity

A realistic portfolio with five assets that have different volatilities and pairwise correlations.

In [ ]:
# Asset labels and annualised volatilities
assets = ["US Equity", "Intl Equity", "Corp Bond", "Gov Bond", "Commodity"]
vols   = np.array([0.18, 0.20, 0.08, 0.05, 0.22])   # annualised std dev

# Correlation matrix
corr = np.array([
    [1.00,  0.80,  0.20, -0.10,  0.15],
    [0.80,  1.00,  0.15, -0.15,  0.20],
    [0.20,  0.15,  1.00,  0.60,  0.05],
    [-0.10, -0.15,  0.60,  1.00, -0.05],
    [0.15,  0.20,  0.05, -0.05,  1.00],
])

# Build covariance matrix: C_ij = corr_ij * σ_i * σ_j
cov5 = corr * np.outer(vols, vols)

result5 = risk_parity(cov5)
print("Portfolio weights:")
for name, w, rc in zip(assets, result5.weights, result5.risk_contributions):
    print(f"  {name:<15s}  weight={w:.4f}  RC={rc:.4f}")
print(f"\nIterations: {result5.iterations}   Converged: {result5.converged}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
x = np.arange(len(assets))

# Portfolio weights
axes[0].bar(x, result5.weights * 100, color=colors)
axes[0].set_xticks(x); axes[0].set_xticklabels(assets, rotation=20, ha="right")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].set_title("Portfolio Weights"); axes[0].set_ylabel("Weight (%)")

# Risk contributions
axes[1].bar(x, result5.risk_contributions * 100, color=colors)
axes[1].axhline(100 / len(assets), color="black", linestyle="--", linewidth=1, label="Equal (20%)")
axes[1].set_xticks(x); axes[1].set_xticklabels(assets, rotation=20, ha="right")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title("Risk Contributions"); axes[1].set_ylabel("Risk Contribution (%)")
axes[1].legend()

fig.suptitle("Five-Asset Risk Parity Portfolio", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 3. Custom risk budgets

Risk parity generalises to **risk budgeting**: assign each asset a target fraction of total portfolio risk.  
Here we allocate 50% of risk to bonds and split the rest equally.

In [ ]:
# 50% risk to Gov Bond, 50% split equally across the other four
b = np.array([0.125, 0.125, 0.125, 0.500, 0.125])
print("Target risk budgets:", b)

result_rb = risk_parity(cov5, b=b)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(x, result_rb.weights * 100, color=colors)
axes[0].set_xticks(x); axes[0].set_xticklabels(assets, rotation=20, ha="right")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].set_title("Portfolio Weights"); axes[0].set_ylabel("Weight (%)")

axes[1].bar(x, result_rb.risk_contributions * 100, color=colors, label="Actual RC")
axes[1].scatter(x, b * 100, color="black", zorder=5, label="Target budget")
axes[1].set_xticks(x); axes[1].set_xticklabels(assets, rotation=20, ha="right")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title("Risk Contributions vs Target"); axes[1].set_ylabel("Risk Contribution (%)")
axes[1].legend()

fig.suptitle("Risk Budgeting: 50% to Gov Bond", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Standalone `risk_contributions` utility

You can compute the risk contributions for **any** portfolio (not just risk parity) using the helper function.

In [ ]:
# Compare equal-weight vs risk-parity portfolio
w_equal = np.ones(5) / 5
rc_equal = risk_contributions(w_equal, cov5)

print("Equal-weight portfolio risk contributions:")
for name, rc in zip(assets, rc_equal):
    print(f"  {name:<15s}  {rc:.4f}  ({rc*100:.1f}%)")

print()
print("Risk-parity portfolio risk contributions:")
for name, rc in zip(assets, result5.risk_contributions):
    print(f"  {name:<15s}  {rc:.4f}  ({rc*100:.1f}%)")

---
## 5. Convergence speed: improved CCD vs portfolio size

The improved CCD method scales as $O(N)$ in computation time. Here we measure iterations across different portfolio sizes.

In [ ]:
import time
from scipy.stats import random_correlation

rng = np.random.default_rng(42)
sizes = [10, 25, 50, 100, 200, 400]
times, iters = [], []

for N in sizes:
    eigs = rng.uniform(0, 1, N); eigs /= eigs.sum() / N
    R = random_correlation.rvs(eigs, random_state=rng)
    vols_n = rng.uniform(0.05, 0.30, N)
    cov_n = R * np.outer(vols_n, vols_n)

    t0 = time.perf_counter()
    res = risk_parity(cov_n)
    times.append((time.perf_counter() - t0) * 1000)
    iters.append(res.iterations)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(sizes, times, "o-", color="#4C72B0")
axes[0].set_xlabel("Number of assets (N)"); axes[0].set_ylabel("CPU time (ms)")
axes[0].set_title("Computation Time")

axes[1].plot(sizes, iters, "s-", color="#DD8452")
axes[1].set_xlabel("Number of assets (N)"); axes[1].set_ylabel("Iterations")
axes[1].set_title("Number of Iterations")

fig.suptitle("Improved CCD Scaling with Portfolio Size", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()